# MorphIQ SDK for Python — Regime identification

Send at least 512 comparable positive prices, ordered oldest to newest. V1 selects the latest 512 and consumes their latest 507 prices; every supplied price is validated before cropping. No sampling interval is inferred. Use split-adjusted closes without cash-dividend total-return adjustment. The optional as_of is the final price's RFC 3339 timestamp with an explicit UTC offset; omission means unknown, not the server clock.

Read each head's status before accessing its prediction. An unresolved decision is available; not_applicable and unavailable have reasons and no prediction. Scores are uncalibrated softmax values, not probabilities, p-values or confidence intervals. Never multiply heads. LOWER_VOL/HIGHER_VOL are relative to the reference model's other state, not historical norms or price direction. The basis describes the prediction target; it does not mean a reference fit ran for this request.

For async Python usage, run python -m examples.identify_regime_async. That standalone script shows the same request with AsyncMorphIQ and await.

The download cell fetches real SPY daily prices from Yahoo through yfinance. Enter a ticker at the prompt (SPY is the default). Docker includes yfinance; elsewhere install the notebooks extra. The download keeps prices in memory; the next cell passes them directly to one regime request. [Yahoo price adjustment semantics](https://ranaroussi.github.io/yfinance/advanced/price_adjustment.html). This notebook runs independently with the SDK and Jupyter installed; it does not import repository helpers.

Run All makes **1 API request**, which may consume your API allowance. Opening the notebook makes no API calls.

[Back to the quickstart](../start-here.ipynb)


In [ ]:
import sys

import morphiq_sdk

print("Python:", sys.version.split()[0])
print("MorphIQ SDK:", morphiq_sdk.__version__)

## Connect to your API

Get your API key from the
[MorphIQ Labs developer portal](https://developers.morphiqlabs.com).
The SDK connects to `https://api.morphiqlabs.com` automatically. Before starting
Jupyter or Docker Compose, set your API key in the terminal:

```sh
export MORPHIQ_API_KEY="YOUR_API_KEY"
```

Docker Compose passes the key into the notebook container. If it is unset,
the cell below prompts. The key prompt is masked: **do not paste your key into a code
cell**. The key stays in this kernel's process environment, not notebook source
or outputs; clear it in the cleanup cell and shut down the kernel when finished.

For a local or controlled test service, set `MORPHIQ_API_BASE_URL` before startup
to override the default API URL.

Each example cell contains a complete example: request inputs, API call, response
fields and error handling. The code is copied from the executable Python scripts.
The server validates input domains and performs pricing or regime inference.


In [ ]:
import os
from getpass import getpass

if not os.environ.get("MORPHIQ_API_KEY"):
    os.environ["MORPHIQ_API_KEY"] = getpass("MorphIQ API key: ")

In [ ]:
from datetime import date, timedelta

import yfinance as yf

ticker = input("Ticker symbol [SPY]: ").strip().upper() or "SPY"
# End is exclusive: omit today's potentially incomplete daily bar.
end = date.today()
history = yf.download(
    ticker,
    start=end - timedelta(days=3 * 366),
    end=end,
    interval="1d",
    auto_adjust=False,  # Close is split-adjusted; Adj Close also adjusts dividends.
    keepna=True,
    multi_level_index=False,
    progress=False,
    threads=False,
)
if history is None or history.empty:
    raise SystemExit("Yahoo returned no price history.")
closes = history[["Close"]].tail(512)
if len(closes) < 512 or closes["Close"].isna().any() or (closes["Close"] <= 0).any():
    raise SystemExit("Need 512 consecutive valid daily closes; no rows were filled or dropped.")
if not closes.index.is_monotonic_increasing or not closes.index.is_unique:
    raise SystemExit("Expected unique daily dates ordered oldest to newest.")
print(f"{ticker}: {len(closes)} split-adjusted daily closes")
print(closes.tail().to_string())

In [ ]:
import os
import sys
from getpass import getpass

from morphiq_sdk import APIError, MorphIQ, RegimeRequest, ResponseError, TransportError

api_key = os.environ.get("MORPHIQ_API_KEY") or getpass("MorphIQ API key: ")
request = RegimeRequest(prices=closes["Close"].tolist())
# Daily date labels do not establish exact close timestamps; omit as_of.
print(f"Sending {len(request.prices)} closes: {closes.index[0]} through {closes.index[-1]}")

try:
    # The client uses https://api.morphiqlabs.com and closes its connection pool here.
    with MorphIQ(api_key=api_key) as client:
        response = client.regime(request)
except APIError as error:
    print(f"HTTP {error.status_code}: {error.code}: {error.message}", file=sys.stderr)
    print(f"Request ID: {error.request_id}", file=sys.stderr)
    charged = error.metadata.fcu_charged if error.metadata else None
    print(
        f"Commercial CU charged: {charged if charged is not None else 'not reported'}",
        file=sys.stderr,
    )
    raise SystemExit(1) from None
except TransportError:
    # Includes timeouts. The service may already have committed a charge.
    raise SystemExit(
        "No complete response; execution and charges may be unknown. No retry."
    ) from None
except ResponseError as error:
    raise SystemExit(
        f"Invalid response; request_id={error.request_id}. Charges may be unknown. No retry."
    ) from None

# Each head has its own status. Unresolved is an available decision label;
# not_applicable and unavailable instead have a reason and no prediction.
for name, head in (
    ("Decision", response.result.decision),
    ("Direction", response.result.direction),
    ("State", response.result.state),
):
    print(f"{name}: {head.status} (target: {head.basis.target})")
    if head.status == "available":
        print(f"  Label: {head.prediction.label}")
        for score in head.prediction.scores:
            print(f"  {score.label}: {score.value}")
    else:
        print(f"  Reason: {head.reason_code}")

# Scores are uncalibrated softmax values, not probabilities or p-values.
# Do not multiply heads. LOWER_VOL/HIGHER_VOL are relative model states,
# not historical volatility levels or price directions.
print("Score kind:", response.result.score_contract.kind)
metadata = response.metadata
consumed = metadata.consumed_price_range
print("Prices received:", metadata.received_price_count)
print("Window size:", metadata.window_price_count)
print(f"Consumed prices: [{consumed.start_index}, {consumed.end_index_exclusive})")
print(f"Selected input price index: {metadata.input_price_index}; as_of: {metadata.as_of}")
print(f"Model: {metadata.provenance.model_id} ({metadata.provenance.model_status})")
print("Model SHA-256:", metadata.provenance.model_sha256)
print("Reference teacher executed:", metadata.provenance.teacher_executed)
print("Request ID:", metadata.request_id)
for engine in metadata.engines:
    print(f"Engine: {engine.name} {engine.version} ({engine.revision})")
charged = metadata.fcu_charged
print("Commercial CU charged:", charged if charged is not None else "not reported")

## Reading results and failures

Use the decision, direction and state heads in `response.result`. Keep `response.metadata.request_id` for support. Commercial charges are `metadata.fcu_charged`; missing charges are **unreported**, not zero. The consumed range is half-open in the original array. The server reports model provenance separately from the engine identity.

`APIError` exposes HTTP status, server code, message, request ID, optional charge
metadata and `Retry-After`. `RequestTimeout` means execution/charges may be
unknown. Both clients make **no automatic retries**, including after timeouts,
and do not follow redirects. Correct the input/key/endpoint or inspect the
reported charge before deliberately submitting another request.

## Finish

The next cell removes the API key from this kernel's environment. Shut down the
kernel to release its remaining memory. If you exported the key in your terminal,
run `unset MORPHIQ_API_KEY` there when finished; `docker compose down` removes the
container configured with that key. Clear outputs before sharing a notebook.


In [ ]:
if "MORPHIQ_API_KEY" in os.environ:
    del os.environ["MORPHIQ_API_KEY"]
print("API key removed from the kernel environment.")